# Randomize score

Load the latest timestamped KBStats player CSV, add a random score from 1 to 150, and save a timestamped CSV.

## Project paths

In [1]:
# Import the libraries required by this notebook step.
import sys
from pathlib import Path


# Handle project root for reuse in the workflow.
def _locate_project_root() -> Path:
    starts = []
    vscode_notebook = globals().get("__vsc_ipynb_file__")
    if isinstance(vscode_notebook, str) and vscode_notebook.strip():
        starts.append(Path(vscode_notebook).expanduser().resolve().parent)
    starts.append(Path.cwd().resolve())

    checked = set()
    # Process each available item while preserving the current workflow state.
    for start in starts:
        # Process each available item while preserving the current workflow state.
        for candidate in (start, *start.parents):
            if candidate in checked:
                continue
            checked.add(candidate)
            if (candidate / "project_paths.py").is_file():
                return candidate
    raise FileNotFoundError(
        "Could not locate project_paths.py. Start Jupyter from the Kickbase "
        "project root or open this notebook from within that project."
    )


# Set workflow configuration value: _PROJECT_ROOT.
_PROJECT_ROOT = _locate_project_root()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from project_paths import (
    EXPECTED_POINTS_DIR,
    KBSTATS_PLAYERS_DIR,
    ensure_directory,
)

## Select the latest KBStats player CSV

In [2]:
# Import the libraries required by this notebook step.
from __future__ import annotations

import random
import re
import warnings
from datetime import datetime, timezone

import pandas as pd
from IPython.display import display

# Set workflow configuration value: SCORE_COLUMN.
SCORE_COLUMN = "score"
# Set workflow configuration value: MINIMUM_SCORE.
MINIMUM_SCORE = 1
# Set workflow configuration value: MAXIMUM_SCORE.
MAXIMUM_SCORE = 150

# Set workflow configuration value: KBSTATS_FILENAME_RE.
KBSTATS_FILENAME_RE = re.compile(
    r"^kbstats_players_"
    r"(?P<timestamp>\d{8}_\d{6}_[+-]\d{4})\.csv$"
)

In [3]:
# Parse and validate kbstats filename for reuse in the workflow.
def parse_kbstats_filename(path: Path) -> tuple[datetime, str]:
    match = KBSTATS_FILENAME_RE.fullmatch(path.name)
    # Validate the input before continuing with later processing.
    if match is None:
        raise ValueError(
            f"Filename does not contain a supported KBStats timestamp: {path.name}"
        )

    timestamp_text = match.group("timestamp")
    # Handle expected failures with a clear, actionable message.
    try:
        parsed = datetime.strptime(timestamp_text, "%Y%m%d_%H%M%S_%z")
    except ValueError as exc:
        raise ValueError(f"Invalid timestamp in {path.name}: {exc}") from exc
    return parsed.astimezone(timezone.utc), timestamp_text


# Select latest kbstats csv for reuse in the workflow.
def select_latest_kbstats_csv(directory: Path) -> tuple[Path, str]:
    # Validate the input before continuing with later processing.
    if not directory.is_dir():
        raise FileNotFoundError(
            f"KBStats player output directory not found: {directory}. "
            "Run the KBStats player extraction notebook first."
        )

    candidates = sorted(directory.glob("kbstats_players_*.csv"))
    # Validate the input before continuing with later processing.
    if not candidates:
        raise FileNotFoundError(
            f"No kbstats_players_*.csv files were found in {directory}."
        )

    parsed_candidates: list[tuple[datetime, Path, str]] = []
    # Process each available item while preserving the current workflow state.
    for path in candidates:
        # Handle expected failures with a clear, actionable message.
        try:
            timestamp, timestamp_text = parse_kbstats_filename(path)
            parsed_candidates.append((timestamp, path, timestamp_text))
        except ValueError as exc:
            warnings.warn(f"Ignoring {path.name}: {exc}", stacklevel=2)

    # Validate the input before continuing with later processing.
    if not parsed_candidates:
        raise ValueError(
            "KBStats CSV files were found, but none had a valid timezone-aware "
            "timestamp in the filename."
        )

    latest_timestamp = max(timestamp for timestamp, _, _ in parsed_candidates)
    latest_candidates = [
        (path, timestamp_text)
        for timestamp, path, timestamp_text in parsed_candidates
        if timestamp == latest_timestamp
    ]
    # Validate the input before continuing with later processing.
    if len(latest_candidates) != 1:
        names = ", ".join(path.name for path, _ in latest_candidates)
        raise RuntimeError(
            "Multiple KBStats CSV files encode the same latest instant: " + names
        )
    return latest_candidates[0]


selected_input_path, source_timestamp = select_latest_kbstats_csv(
    KBSTATS_PLAYERS_DIR
)
print(f"Selected input: {selected_input_path}")
print(f"Source timestamp: {source_timestamp}")

Selected input: C:\kickbase project\outputs\kbstats\players\kbstats_players_20260817_194320_+0200.csv
Source timestamp: 20260817_194320_+0200


## Add randomized score

In [4]:
# Handle expected failures with a clear, actionable message.
try:
    players_df = pd.read_csv(selected_input_path, encoding="utf-8-sig")
except pd.errors.EmptyDataError as exc:
    raise ValueError(f"Input CSV is empty: {selected_input_path}") from exc
except (OSError, UnicodeError, pd.errors.ParserError) as exc:
    raise ValueError(f"Could not read input CSV {selected_input_path}: {exc}") from exc

# Validate the input before continuing with later processing.
if players_df.empty:
    raise ValueError(f"Input CSV contains no player rows: {selected_input_path}")
# Validate the input before continuing with later processing.
if SCORE_COLUMN in players_df.columns:
    raise ValueError(
        f"Input CSV already contains a {SCORE_COLUMN!r} column; "
        "refusing to overwrite it."
    )

random_source = random.SystemRandom()
players_df[SCORE_COLUMN] = [
    random_source.randint(MINIMUM_SCORE, MAXIMUM_SCORE)
    for _ in range(len(players_df))
]

print(f"Rows: {len(players_df):,}")
print(
    f"Generated range: {players_df[SCORE_COLUMN].min()} to "
    f"{players_df[SCORE_COLUMN].max()}"
)
display(players_df.head())

Rows: 468
Generated range: 1 to 150


,id,teamId,firstName,lastName,marketValue,averagePoints,games,gamesPlayed,start11,totalPlaytimeS,...,playerImage,number,trend,pointsPerValue,pointsStdDev,history,startProbability,name,marketChangeToday,expected points
0,173,2,Jonathan,Tah,36786174.0,127.0,28.0,28.0,23.0,135209.0,...,https://kickbase.b-cdn.net/content/file/bbdbcc...,4.0,1,96.0,NaN,"[{""hasPlayed"": true, ""points"": 146}, {""hasPlay...",1,Jonathan Tah,48730.0,67
1,237,2,Manuel,Neuer,12761537.0,95.0,22.0,22.0,22.0,124205.0,...,https://kickbase.b-cdn.net/content/file/aadd45...,1.0,1,163.0,NaN,"[{""hasPlayed"": true, ""points"": 45}, {""hasPlaye...",2,Manuel Neuer,70077.0,40
2,383,2,Sven,Ulreich,1858016.0,125.0,1.0,1.0,1.0,6244.0,...,https://kickbase.b-cdn.net/content/file/903ab6...,26.0,2,67.0,NaN,"[{""hasPlayed"": false, ""points"": null}, {""hasPl...",5,Sven Ulreich,-210462.0,75
3,1685,2,Joshua,Kimmich,59676168.0,186.0,29.0,29.0,24.0,151968.0,...,https://kickbase.b-cdn.net/content/file/e84941...,6.0,1,90.0,NaN,"[{""hasPlayed"": true, ""points"": 317}, {""hasPlay...",1,Joshua Kimmich,27719.0,42
4,1961,2,Serge,Gnabry,17314237.0,142.0,21.0,21.0,16.0,80707.0,...,https://kickbase.b-cdn.net/content/file/e88435...,7.0,2,172.0,NaN,"[{""hasPlayed"": false, ""points"": null}, {""hasPl...",5,Serge Gnabry,-413437.0,44


## Save the result

In [5]:
created_datetime = datetime.now().astimezone()
creation_timestamp = created_datetime.strftime("%Y%m%d_%H%M%S_%z")
output_directory = ensure_directory(EXPECTED_POINTS_DIR)
output_path = output_directory / (
    f"expected_points_{source_timestamp}_random_{creation_timestamp}.csv"
)

# Handle expected failures with a clear, actionable message.
try:
    players_df.to_csv(output_path, index=False, encoding="utf-8-sig")
except OSError as exc:
    raise OSError(f"Could not write score CSV {output_path}: {exc}") from exc

print(f"Output: {output_path}")
print(f"Created at: {created_datetime.isoformat(timespec='seconds')}")

Output: C:\kickbase project\outputs\expected_points\expected_points_20260817_194320_+0200_random_20260818_101310_+0200.csv
Created at: 2026-08-18T10:13:10+02:00
